# 🚀 Underwater Image Enhancement - Local RTX Training & Evaluation Pipeline
### Khảo sát tuần tự các Variants siêu nhẹ (< 200k params) trên `EUVP Dark + UIEB`
Notebook này được thiết kế để chạy toàn diện trên máy tính cá nhân (GPU RTX 8GB VRAM), với các tính năng:
1. **Khảo sát tuần tự 3 Variants:**
   - **V1: LiteEnhanceNet** (Bản gốc chính thức từ `zhangsong1213/LiteEnhanceNet` ~13.7k params)
   - **V2: PLite-Net** (LiteEnhanceNet + Ràng buộc vật lý tự suy biến quang học xuôi $I_{\text{redeg}} = J \cdot t + B(1-t)$ ~18.4k params)
   - **V3: PLCS-Lite** (Mô hình lai đề xuất: LCCM + SMSDB + Depthwise OSA + Physical Re-degradation ~101.3k params)
2. **Bộ dữ liệu:** Huấn luyện & Valid trên `EUVP underwater_dark + UIEB`, và tự động Test trên **cả 2 bộ**: `EUVP Dark Test` & `UIEB T90 Test`.
3. **Theo dõi chi tiết:** In rõ ràng từng **Epoch, Thời gian thực thi (s/epoch), Thời gian còn lại (ETA), Train Loss, Val PSNR/SSIM**, và lưu tự động `best_model.pth`.


In [ ]:
import os
import sys
import time
import math
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Thêm source code uwir vào sys.path
PROJECT_ROOT = Path('.').resolve()
sys.path.append(str(PROJECT_ROOT / 'src'))

# 1. Kiểm tra phần cứng GPU RTX
print('=' * 65)
print('THÔNG TIN PHẦN CỨNG & MÔI TRƯỜNG HUẤN LUYỆN')
print('=' * 65)
print(f'PyTorch Version : {torch.__version__}')
cuda_avail = torch.cuda.is_available()
print(f'CUDA Available  : {cuda_avail}')
if cuda_avail:
    device = torch.device('cuda:0')
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'Active GPU      : {gpu_name} ({vram_gb:.2f} GB VRAM)')
else:
    device = torch.device('cpu')
    print('CẢNH BÁO: Chưa tìm thấy GPU CUDA, đang chạy bằng CPU!')
print('=' * 65)


In [ ]:
# 2. CẤU HÌNH ĐƯỜNG DẪN DỮ LIỆU
# Bạn có thể chỉnh lại đường dẫn cho đúng với ổ đĩa trên máy bàn của bạn:
EUVP_DATA_DIR = r"D:\Dataset\EUVP"       # Chứa Paired/underwater_dark/trainA & trainB
UIEB_DATA_DIR = r"D:\Dataset\UIEB"       # Chứa raw-890 và reference-890

# Cấu hình siêu tham số
IMG_SIZE = 256
BATCH_SIZE = 16
NUM_WORKERS = 4
EPOCHS = 50
LEARNING_RATE = 3e-4
DEVICE = device
USE_AMP = cuda_avail  # Kích hoạt Mixed Precision giúp GPU mát và chạy siêu nhanh

print('Cấu hình huấn luyện:')
print(f' - Image Size : {IMG_SIZE}x{IMG_SIZE}')
print(f' - Batch Size : {BATCH_SIZE}')
print(f' - Epochs     : {EPOCHS}')
print(f' - Mixed Prec : {USE_AMP}')


In [ ]:
from uwir.models import build_model, ALL_MODEL_NAMES

# 3. KIỂM TRA THÔNG SỐ CÁC MÔ HÌNH THỬ NGHIỆM
models_to_test = [
    ('Variant 1: LiteEnhanceNet (Official Baseline)', 'lite_enhancenet_3ch'),
    ('Variant 2: PLite-Net (Physical Re-degradation)', 'plite_3ch'),
    ('Variant 3: PLCS-Lite (Proposed Hybrid SOTA)', 'plcs_lite_3ch'),
]

print(f"{'Variant':<45} | {'Params':>12} | {'Trọng lượng':>12}")
print('-' * 75)
for title, name in models_to_test:
    model = build_model(name)
    p_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    size_mb = p_count * 4 / (1024 * 1024)
    print(f"{title:<45} | {p_count:>10,} | {size_mb:>10.2f} MB")
print('-' * 75)


In [ ]:
from uwir.data.factory import get_euvp_training_set, get_uieb_training_set
from uwir.cli.train import _split_train_validation

print('Đang chuẩn bị bộ dữ liệu EUVP Dark + UIEB...')
try:
    euvp_ds = get_euvp_training_set(EUVP_DATA_DIR, img_size=IMG_SIZE, subset='underwater_dark')
    uieb_ds = get_uieb_training_set(UIEB_DATA_DIR, img_size=IMG_SIZE)
    combined_ds = torch.utils.data.ConcatDataset([euvp_ds, uieb_ds])
    
    # Tách 90% Train / 10% Valid có paired ground truth
    train_ds, val_ds = _split_train_validation(combined_ds, seed=42)
    
    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
        num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda')
    )
    val_loader = torch.utils.data.DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
        num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda')
    )
    print(f'✅ Khởi tạo thành công: {len(train_ds)} train samples, {len(val_ds)} val samples.')
except Exception as e:
    print('⚠️ Chưa tìm thấy đúng thư mục dữ liệu thật. Vui lòng cập nhật EUVP_DATA_DIR và UIEB_DATA_DIR ở Cell 2!')
    print('Chi tiết lỗi:', e)


In [ ]:
from uwir.losses import CompositeLoss
from uwir.metrics import compute_psnr, compute_ssim

def format_time(seconds):
    mins, secs = divmod(int(seconds), 60)
    hours, mins = divmod(mins, 60)
    if hours > 0:
        return f'{hours}h {mins:02d}m {secs:02d}s'
    return f'{mins:02d}m {secs:02d}s'

def train_variant(model_name, run_title, epochs=50, lr=3e-4, lambda_redeg=0.5):
    print('=' * 80)
    print(f'BẮT ĐẦU HUẤN LUYỆN: {run_title} ({model_name})')
    print('=' * 80)
    
    # 1. Khởi tạo mô hình
    model = build_model(model_name).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    
    # 2. Hàm Loss: Hỗ trợ cả Task Loss + Re-degradation Loss
    criterion = CompositeLoss(
        lambda_l1=1.0, lambda_perc=0.1, lambda_ssim=0.5,
        lambda_hsvcs=0.1, lambda_redeg=lambda_redeg, device=DEVICE
    )
    
    ckpt_dir = Path('checkpoints') / model_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_psnr = -1.0
    history = {'train_loss': [], 'val_psnr': [], 'val_ssim': [], 'epoch_times': []}
    
    total_start = time.time()
    
    for epoch in range(1, epochs + 1):
        epoch_start = time.time()
        model.train()
        tot_loss, l1_acc, ssim_acc, redeg_acc = 0.0, 0.0, 0.0, 0.0
        
        for batch_idx, (inp, gt, _, _) in enumerate(train_loader):
            inp = inp.to(DEVICE, non_blocking=True)
            gt = gt.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            
            with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                pred = model(inp)
                loss, parts = criterion(pred, gt, input_image=inp)
                
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            
            tot_loss += loss.item()
            l1_acc += parts.get('l1', 0.0)
            ssim_acc += parts.get('ssim_loss', 0.0)
            redeg_acc += parts.get('redeg', 0.0)
            
        scheduler.step()
        n_batches = len(train_loader)
        avg_train_loss = tot_loss / n_batches
        epoch_time = time.time() - epoch_start
        history['epoch_times'].append(epoch_time)
        history['train_loss'].append(avg_train_loss)
        
        # Validation
        model.eval()
        val_psnr_list, val_ssim_list = [], []
        with torch.no_grad():
            for inp, gt, _, _ in val_loader:
                inp = inp.to(DEVICE, non_blocking=True)
                gt = gt.to(DEVICE, non_blocking=True)
                with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    pred = model(inp)
                    if isinstance(pred, (tuple, list)):
                        pred = pred[0]
                pred = pred.clamp(0.0, 1.0)
                val_psnr_list.append(compute_psnr(pred, gt).item())
                val_ssim_list.append(compute_ssim(pred, gt).item())
                
        avg_psnr = np.mean(val_psnr_list)
        avg_ssim = np.mean(val_ssim_list)
        history['val_psnr'].append(avg_psnr)
        history['val_ssim'].append(avg_ssim)
        
        # Tính toán thời gian còn lại (ETA)
        elapsed = time.time() - total_start
        avg_epoch_time = np.mean(history['epoch_times'])
        remaining_time = avg_epoch_time * (epochs - epoch)
        
        is_best = avg_psnr > best_psnr
        if is_best:
            best_psnr = avg_psnr
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'best_psnr': best_psnr,
                'val_ssim': avg_ssim,
            }, ckpt_dir / 'best_model.pth')
            tag = f'🌟 [BEST PSNR: {best_psnr:.2f} dB]'
        else:
            tag = ''
            
        # In chi tiết từng epoch
        redeg_str = f", Redeg: {redeg_acc/n_batches:.3f}" if lambda_redeg > 0 else ""
        print(
            f"Epoch [{epoch:2d}/{epochs}] "
            f"({epoch_time:4.1f}s | ETA: {format_time(remaining_time)}) | "
            f"Loss: {avg_train_loss:.4f} (L1: {l1_acc/n_batches:.3f}, SSIM: {ssim_acc/n_batches:.3f}{redeg_str}) | "
            f"Val PSNR: {avg_psnr:.2f} dB, SSIM: {avg_ssim:.4f} {tag}"
        )
        
    print(f"\n✅ Hoàn tất huấn luyện {model_name} trong {format_time(time.time() - total_start)}!")
    print(f"🏆 Best Validation PSNR: {best_psnr:.2f} dB (Đã lưu tại {ckpt_dir / 'best_model.pth'})\n")
    return history


In [ ]:
# 5. CHẠY THỰC NGHIỆM TUẦN TỰ 3 VARIANTS

# Variant 1: LiteEnhanceNet (Baseline gốc chính thức)
hist_v1 = train_variant('lite_enhancenet_3ch', 'Variant 1: LiteEnhanceNet Gốc', epochs=EPOCHS, lr=LEARNING_RATE, lambda_redeg=0.0)

# Variant 2: PLite-Net (Thêm ràng buộc vật lý tự suy biến quang học)
hist_v2 = train_variant('plite_3ch', 'Variant 2: PLite-Net (Vật lý tự suy biến)', epochs=EPOCHS, lr=LEARNING_RATE, lambda_redeg=0.5)

# Variant 3: PLCS-Lite (Mô hình lai đề xuất: LCCM + SMSDB + Depthwise + Vật lý)
hist_v3 = train_variant('plcs_lite_3ch', 'Variant 3: PLCS-Lite (Mô hình lai đề xuất)', epochs=EPOCHS, lr=LEARNING_RATE, lambda_redeg=0.5)


In [ ]:
# 6. BIỂU ĐỒ SO SÁNH QUÁ TRÌNH HỘI TỤ (PSNR & LOSS)
plt.figure(figsize=(14, 5))

# Plot PSNR
plt.subplot(1, 2, 1)
if 'hist_v1' in locals(): plt.plot(hist_v1['val_psnr'], label='V1: LiteEnhanceNet (13.7k)', color='gray', linestyle='--')
if 'hist_v2' in locals(): plt.plot(hist_v2['val_psnr'], label='V2: PLite-Net (+Physical 18.4k)', color='orange')
if 'hist_v3' in locals(): plt.plot(hist_v3['val_psnr'], label='V3: PLCS-Lite (Proposed 101.3k)', color='green', linewidth=2)
plt.title('Validation PSNR theo từng Epoch (dB)')
plt.xlabel('Epoch')
plt.ylabel('PSNR (dB)')
plt.grid(True, linestyle=':')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
if 'hist_v1' in locals(): plt.plot(hist_v1['train_loss'], label='V1 Loss', color='gray', linestyle='--')
if 'hist_v2' in locals(): plt.plot(hist_v2['train_loss'], label='V2 Loss', color='orange')
if 'hist_v3' in locals(): plt.plot(hist_v3['train_loss'], label='V3 Loss', color='green', linewidth=2)
plt.title('Training Loss theo từng Epoch')
plt.xlabel('Epoch')
plt.ylabel('Total Loss')
plt.grid(True, linestyle=':')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
from uwir.cli.evaluate import collect_test_pairs, collect_uieb_test_pairs, TestDataset
from uwir.metrics import evaluate_loader

# 7. ĐÁNH GIÁ ĐỘC LẬP TRÊN CẢ 2 BỘ TEST (EUVP Dark Test & UIEB T90 Test)
print('=' * 75)
print('ĐÁNH GIÁ CHECKPOINTS TRÊN CẢ 2 BỘ TEST BENCHMARKS')
print('=' * 75)

# Tải 2 bộ test
euvp_test_pairs = collect_test_pairs(EUVP_DATA_DIR)
uieb_test_pairs = collect_uieb_test_pairs(UIEB_DATA_DIR, seed=42, count=90)

print(f'Số lượng mẫu test EUVP Dark : {len(euvp_test_pairs)}')
print(f'Số lượng mẫu test UIEB T90  : {len(uieb_test_pairs)}')

def evaluate_checkpoint(ckpt_path, model_name, pairs, benchmark_name):
    if not os.path.exists(ckpt_path):
        return {'psnr': 0.0, 'ssim': 0.0}
    model = build_model(model_name).to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    
    ds = TestDataset(pairs, img_size=256)
    loader = torch.utils.data.DataLoader(ds, batch_size=8, shuffle=False)
    metrics = evaluate_loader(model, loader, device=DEVICE)
    return metrics

results = []
for title, name in models_to_test:
    ckpt_path = Path('checkpoints') / name / 'best_model.pth'
    m_euvp = evaluate_checkpoint(ckpt_path, name, euvp_test_pairs, 'EUVP Dark')
    m_uieb = evaluate_checkpoint(ckpt_path, name, uieb_test_pairs, 'UIEB T90')
    results.append((title, m_euvp, m_uieb))

print('\n' + '=' * 85)
print(f"{'Model Variant':<40} | {'EUVP Dark PSNR / SSIM':^20} | {'UIEB T90 PSNR / SSIM':^20}")
print('=' * 85)
for title, m_euvp, m_uieb in results:
    euvp_str = f"{m_euvp['psnr']:.2f} dB / {m_euvp['ssim']:.4f}" if m_euvp['psnr'] > 0 else 'N/A'
    uieb_str = f"{m_uieb['psnr']:.2f} dB / {m_uieb['ssim']:.4f}" if m_uieb['psnr'] > 0 else 'N/A'
    print(f"{title:<40} | {euvp_str:^20} | {uieb_str:^20}")
print('=' * 85)


In [ ]:
# 8. HIỂN THỊ ẢNH KẾT QUẢ TRỰC QUAN (BEFORE / AFTER)
if len(uieb_test_pairs) > 0:
    sample_inp_path, sample_gt_path = uieb_test_pairs[0]
    from PIL import Image
    import torchvision.transforms as T
    
    img_in = Image.open(sample_inp_path).convert('RGB')
    img_gt = Image.open(sample_gt_path).convert('RGB')
    tensor_in = T.ToTensor()(T.Resize((256, 256))(img_in)).unsqueeze(0).to(DEVICE)
    
    plt.figure(figsize=(15, 4))
    plt.subplot(1, 5, 1); plt.imshow(img_in); plt.title('Raw Input'); plt.axis('off')
    
    col = 2
    for title, name in models_to_test:
        ckpt_path = Path('checkpoints') / name / 'best_model.pth'
        if ckpt_path.exists():
            m = build_model(name).to(DEVICE)
            m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE)['model_state'])
            m.eval()
            with torch.no_grad():
                out = m(tensor_in)
                if isinstance(out, (tuple, list)): out = out[0]
                out_img = out.squeeze(0).permute(1, 2, 0).cpu().clamp(0, 1).numpy()
            plt.subplot(1, 5, col); plt.imshow(out_img); plt.title(name); plt.axis('off')
            col += 1
            
    plt.subplot(1, 5, 5); plt.imshow(img_gt); plt.title('Ground Truth'); plt.axis('off')
    plt.tight_layout()
    plt.show()
